## ***Data Ingestion & Streaming Algorithms***

### ***Install & Imports***

In [1]:
!pip install pybloom-live azure-storage-blob -q

In [2]:
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import numpy as np
import pandas as pd
import requests
from pybloom_live import BloomFilter

### ***Config***

In [3]:
secrets = UserSecretsClient()
os.environ["AZURE_STORAGE_CONN_STR"] = secrets.get_secret("AZURE_STORAGE_CONN_STR")
os.environ["AZURE_CONTAINER"] = secrets.get_secret("AZURE_CONTAINER")

In [4]:
OUTPUT_DIR = Path("/kaggle/working/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Kraken symbols
KRAKEN_SYMBOLS = {
    "XBTUSD":  "BTC",
    "ETHUSD":  "ETH",
    "SOLUSD":  "SOL",
    "XRPUSD":  "XRP",
    "ADAUSD":  "ADA",
    "DOTUSD":  "DOT",
    "DOGEUSD": "DOGE",
    "LTCUSD":  "LTC",
    "AVAXUSD": "AVAX",
    "LINKUSD": "LINK",
    "UNIUSD":  "UNI",
    "ATOMUSD": "ATOM",
    "XLMUSD":  "XLM",
    "FILUSD":  "FIL",
    "ICPUSD":  "ICP",
    "ETCUSD":  "ETC",
    "ALGOUSD": "ALGO",
    "TRXUSD":  "TRX",
    "AAVEUSD": "AAVE",
    "NEARUSD": "NEAR",
}

KRAKEN_OHLC   = "https://api.kraken.com/0/public/OHLC"
KRAKEN_TICKER = "https://api.kraken.com/0/public/Ticker"

### ***Load Data from Azure Blob***

In [6]:
# Load function for Azure
def load_from_azure() -> list:
    """
    Pull raw_stream.json from Azure Blob Storage.
    Requires Kaggle Secrets: AZURE_STORAGE_CONN_STR and AZURE_CONTAINER.
    Returns list of records, or empty list if Azure is not configured.
    """
    try:
        from azure.storage.blob import BlobServiceClient

        conn_str       = os.environ["AZURE_STORAGE_CONN_STR"]
        container_name = os.environ["AZURE_CONTAINER"]
        client         = BlobServiceClient.from_connection_string(conn_str)
        container      = client.get_container_client(container_name)

        data = container.download_blob("notebook_data/raw_stream.json").readall()
        records = json.loads(data)
        print(f"[Azure] Loaded {len(records):,} records from notebook_data/raw_stream.json")
        return records

    except KeyError:
        print("[Azure] Secrets AZURE_STORAGE_CONN_STR / AZURE_CONTAINER not set — falling back to Kraken")
        return []
    except Exception as e:
        print(f"[Azure] Failed: {e} — falling back to Kraken")
        return []

In [7]:
# OHLC for Kraken fallback
def fetch_kraken_ohlc(pair: str, interval: int = 60, limit: int = 500) -> list:
    """Fetch up to 500 hourly candles from Kraken for one pair."""
    try:
        r = requests.get(KRAKEN_OHLC, params={"pair": pair, "interval": interval}, timeout=15)
        r.raise_for_status()
        data = r.json()
        if data.get("error"):
            print(f"[WARN] Kraken error [{pair}]: {data['error']}")
            return []
        result  = data.get("result", {})
        candles = next((v for k, v in result.items() if k != "last"), [])
        return candles[-limit:]
    except Exception as e:
        print(f"[WARN] Kraken OHLC failed [{pair}]: {e}")
        return []

In [8]:
# Ticker for Kraken fallback
def fetch_kraken_ticker(pair: str) -> dict:
    """Fetch current 24h ticker from Kraken."""
    try:
        r = requests.get(KRAKEN_TICKER, params={"pair": pair}, timeout=10)
        r.raise_for_status()
        data = r.json()
        if data.get("error"):
            return {}
        result = data.get("result", {})
        return next(iter(result.values()), {})
    except Exception:
        return {}

In [9]:
# Kraken fallback load function
def load_from_kraken() -> list:
    """Fetch 500 hourly candles per symbol directly from Kraken."""
    print("Fetching from Kraken (fallback)...")
    all_records = []

    for pair, base in KRAKEN_SYMBOLS.items():
        print(f"{pair}...", end=" ", flush=True)
        candles = fetch_kraken_ohlc(pair)

        if not candles:
            print("skipped")
            continue

        ticker      = fetch_kraken_ticker(pair)
        last_price  = float(ticker.get("c", [0])[0]) if ticker else float(candles[-1][4])
        volume_24h  = float(ticker.get("v", [0, 0])[1]) if ticker else 0
        high_24h    = float(ticker.get("h", [0, 0])[1]) if ticker else 0
        low_24h     = float(ticker.get("l", [0, 0])[1]) if ticker else 0
        open_24h    = float(ticker.get("o", last_price)) if ticker else last_price
        pct_chg_24h = (last_price - open_24h) / open_24h * 100 if open_24h else 0

        for i, c in enumerate(candles):
            try:
                candle_ts = int(c[0])
                record = {
                    "id":                   f"{pair}_{candle_ts}",
                    "symbol":               pair,
                    "base_asset":           base,
                    "candle_index":         i,
                    "open_time":            candle_ts * 1000,
                    "timestamp":            datetime.fromtimestamp(candle_ts, tz=timezone.utc).isoformat(),
                    "open":                 float(c[1]),
                    "high":                 float(c[2]),
                    "low":                  float(c[3]),
                    "close":               float(c[4]),
                    "vwap":                 float(c[5]),
                    "volume":               float(c[6]),
                    "trade_count":          int(c[7]),
                    "quote_volume":         float(c[6]) * float(c[5]),
                    "current_price":        last_price,
                    "price_change_pct_24h": round(pct_chg_24h, 4),
                    "high_24h":             high_24h,
                    "low_24h":              low_24h,
                    "volume_24h":           volume_24h,
                    "quote_volume_24h":     volume_24h * last_price,
                    "market_cap":           0,
                    "circulating_supply":   0,
                    "max_supply":           None,
                    "rank":                 0,
                }
                all_records.append(record)
            except (IndexError, ValueError) as e:
                print(f"[WARN] Skipping malformed candle {i} for {pair}: {e}")
                continue

        print(f"{len(candles)} candles")
        time.sleep(0.5)   # Kraken rate limit

    return all_records

In [10]:
# Main data load
print("Loading data...")
raw_stream = load_from_azure()

if not raw_stream:
    print("Azure not configured or empty. Fetching from Kraken...")
    raw_stream = load_from_kraken()

print(f"\nTotal records loaded: {len(raw_stream):,}")
print(f"Symbols: {sorted(set(r['symbol'] for r in raw_stream))}")

with open(OUTPUT_DIR / "raw_stream.json", "w") as f:
    json.dump(raw_stream, f)
print(f"Saved raw_stream.json to {OUTPUT_DIR}")

Loading data...
[Azure] Loaded 30,650 records from notebook_data/raw_stream.json

Total records loaded: 30,650
Symbols: ['AAVEUSD', 'ADAUSD', 'ALGOUSD', 'ATOMUSD', 'AVAXUSD', 'DOGEUSD', 'DOTUSD', 'ETCUSD', 'ETHUSD', 'FILUSD', 'ICPUSD', 'LINKUSD', 'LTCUSD', 'NEARUSD', 'SOLUSD', 'TRXUSD', 'UNIUSD', 'XBTUSD', 'XLMUSD', 'XRPUSD']
Saved raw_stream.json to /kaggle/working/data


### ***Streaming Algorithm Classes***

In [11]:
# Bloom Filter
class BloomFilterLogger:
    """Detects duplicate records in the stream. Does NOT discard — just flags."""
    def __init__(self, capacity: int = 100_000, error_rate: float = 0.01):
        self.bloom    = BloomFilter(capacity=capacity, error_rate=error_rate)
        self.seen     = 0
        self.dupes    = 0
        self.dupe_ids = []

    def process(self, record_id: str) -> bool:
        self.seen += 1
        if record_id in self.bloom:
            self.dupes += 1
            self.dupe_ids.append(record_id)
            return True
        self.bloom.add(record_id)
        return False

    def stats(self) -> dict:
        return {
            "total_seen":   self.seen,
            "duplicates":   self.dupes,
            "dupe_rate":    round(self.dupes / max(self.seen, 1), 4),
            "sample_dupes": self.dupe_ids[:5],
        }

In [12]:
# Reservoir Sampling
class ReservoirSampler:
    """Algorithm R: uniform random sample of exactly k items from a stream."""
    def __init__(self, k: int = 300):
        self.k         = k
        self.reservoir = []
        self.n         = 0

    def add(self, item: dict) -> None:
        self.n += 1
        if len(self.reservoir) < self.k:
            self.reservoir.append(item)
        else:
            j = random.randint(0, self.n - 1)
            if j < self.k:
                self.reservoir[j] = item

    def sample(self) -> list:
        return list(self.reservoir)

    def stats(self) -> dict:
        return {
            "stream_size":    self.n,
            "reservoir_size": len(self.reservoir),
            "sampling_rate":  round(len(self.reservoir) / max(self.n, 1), 4),
        }

In [13]:
# Differential Privacy
class DifferentialPrivacy:
    """Laplace mechanism: adds calibrated noise to sensitive numeric fields."""
    SENSITIVE_FIELDS = ["volume_24h", "quote_volume_24h", "volume", "quote_volume"]

    def __init__(self, epsilon: float = 1.0):
        self.epsilon   = epsilon
        self.noise_log = []

    def privatise(self, record: dict) -> dict:
        noisy  = record.copy()
        noises = {}
        for field in self.SENSITIVE_FIELDS:
            if field in noisy and noisy[field] is not None:
                val         = float(noisy[field])
                sensitivity = max(abs(val) * 0.001, 1e-6)
                noise       = np.random.laplace(0, sensitivity / self.epsilon)
                noisy[field] = val + noise
                noises[field] = round(noise, 6)
        self.noise_log.append(noises)
        return noisy

    def stats(self) -> dict:
        if not self.noise_log:
            return {}
        df_noise = pd.DataFrame(self.noise_log)
        return {
            "epsilon":             self.epsilon,
            "records_privatised":  len(self.noise_log),
            "avg_abs_noise_per_field": {
                col: round(df_noise[col].abs().mean(), 6)
                for col in df_noise.columns
            },
        }

In [14]:
# Flajolet-Martin
class FlajoletMartin:
    """Estimates distinct ID count without storing all IDs. Uses 10 hash functions."""
    def __init__(self, n_hashes: int = 10):
        self.n_hashes  = n_hashes
        self.max_zeros = [0] * n_hashes

    def add(self, record_id: str) -> None:
        for i in range(self.n_hashes):
            h     = hashlib.sha256(f"{i}:{record_id}".encode()).hexdigest()
            bits  = bin(int(h, 16))[2:]
            zeros = len(bits) - len(bits.rstrip("0"))
            self.max_zeros[i] = max(self.max_zeros[i], zeros)

    def estimate(self) -> int:
        return int(np.median([2 ** z for z in self.max_zeros]))

    def stats(self) -> dict:
        return {
            "estimated_distinct_ids": self.estimate(),
            "max_zeros_per_hash":     self.max_zeros,
        }

### ***Streaming Simulation***

In [15]:
RESERVOIR_K = 300
EPSILON     = 1.0

bloom_filter = BloomFilterLogger(capacity=max(len(raw_stream) * 2, 10_000))
reservoir    = ReservoirSampler(k=RESERVOIR_K)
dp           = DifferentialPrivacy(epsilon=EPSILON)
fm_sketch    = FlajoletMartin(n_hashes=10)

# Inject 5% duplicate records to simulate Kafka consumer group resets
n_dupes           = max(1, int(len(raw_stream) * 0.05))
dupe_records      = random.sample(raw_stream, min(n_dupes, len(raw_stream)))
stream_with_dupes = raw_stream + dupe_records
random.shuffle(stream_with_dupes)

print(f"\nStream size: {len(stream_with_dupes):,} records ({n_dupes} injected duplicates)")

for i, record in enumerate(stream_with_dupes):
    rid = record["id"]
    bloom_filter.process(rid)           # 1. Bloom Filter
    fm_sketch.add(rid)                  # 2. Flajolet-Martin
    private_record = dp.privatise(record)  # 3. Differential Privacy
    reservoir.add(private_record)       # 4. Reservoir Sampling

    if (i + 1) % 2000 == 0:
        print(f"  Processed {i+1:,} | Reservoir: {len(reservoir.reservoir)} | "
              f"Distinct≈{fm_sketch.estimate():,}")

print(f"\nStream processing complete")


Stream size: 32,182 records (1532 injected duplicates)
  Processed 2,000 | Reservoir: 300 | Distinct≈3,072
  Processed 4,000 | Reservoir: 300 | Distinct≈8,192
  Processed 6,000 | Reservoir: 300 | Distinct≈8,192
  Processed 8,000 | Reservoir: 300 | Distinct≈8,192
  Processed 10,000 | Reservoir: 300 | Distinct≈8,192
  Processed 12,000 | Reservoir: 300 | Distinct≈12,288
  Processed 14,000 | Reservoir: 300 | Distinct≈12,288
  Processed 16,000 | Reservoir: 300 | Distinct≈12,288
  Processed 18,000 | Reservoir: 300 | Distinct≈16,384
  Processed 20,000 | Reservoir: 300 | Distinct≈16,384
  Processed 22,000 | Reservoir: 300 | Distinct≈24,576
  Processed 24,000 | Reservoir: 300 | Distinct≈32,768
  Processed 26,000 | Reservoir: 300 | Distinct≈32,768
  Processed 28,000 | Reservoir: 300 | Distinct≈32,768
  Processed 30,000 | Reservoir: 300 | Distinct≈32,768
  Processed 32,000 | Reservoir: 300 | Distinct≈32,768

Stream processing complete


### ***Stats & Save Outputs***

In [16]:
bf_stats = bloom_filter.stats()
rs_stats = reservoir.stats()
dp_stats = dp.stats()
fm_stats = fm_sketch.stats()
actual_distinct = len(set(r["id"] for r in raw_stream))

print(f"Bloom Filter      | Seen: {bf_stats['total_seen']:,} | Dupes: {bf_stats['duplicates']:,} ({bf_stats['dupe_rate']:.1%})")
print(f"Reservoir Sample  | Stream: {rs_stats['stream_size']:,} | Sample: {rs_stats['reservoir_size']} ({rs_stats['sampling_rate']:.1%})")
print(f"Diff. Privacy     | ε={dp_stats.get('epsilon')} | Records noised: {dp_stats.get('records_privatised',0):,}")
print(f"Flajolet-Martin   | Actual distinct: {actual_distinct:,} | Estimated: {fm_stats['estimated_distinct_ids']:,}")

Bloom Filter      | Seen: 32,182 | Dupes: 1,535 (4.8%)
Reservoir Sample  | Stream: 32,182 | Sample: 300 (0.9%)
Diff. Privacy     | ε=1.0 | Records noised: 32,182
Flajolet-Martin   | Actual distinct: 30,650 | Estimated: 32,768


In [17]:
reservoir_sample = reservoir.sample()
with open(OUTPUT_DIR / "reservoir_sample.json", "w") as f:
    json.dump(reservoir_sample, f)

stream_stats = {
    "bloom_filter":         bf_stats,
    "reservoir_sampling":   rs_stats,
    "differential_privacy": dp_stats,
    "flajolet_martin":      fm_stats,
    "actual_distinct_ids":  actual_distinct,
    "total_records":        len(raw_stream),
    "symbols":              sorted(set(r["symbol"] for r in raw_stream)),
}
with open(OUTPUT_DIR / "stream_stats.json", "w") as f:
    json.dump(stream_stats, f, indent=2)

In [18]:
print(f"Output files in {OUTPUT_DIR}:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size/1024:6.1f} KB")


Output files in /kaggle/working/data:
raw_stream.json                     16729.1 KB
reservoir_sample.json                163.7 KB
stream_stats.json                      1.2 KB
